In [14]:
import requests

url = "https://www.iranjib.ir/showarchive/0/0"
response = requests.get(url)
print(response.text)

 <!DOCTYPE html><html dir="rtl" lang="fa"><head>
<meta name="viewport" id="the_viewport" content="width=device-width,initial-scale=1">
<link rel="apple-touch-icon" sizes="180x180" href="/apple-touch-icon.png">
<link rel="icon" type="image/png" sizes="32x32" href="/favicon-32x32.png">
<link rel="icon" type="image/png" sizes="16x16" href="/favicon-16x16.png">
<link rel="mask-icon" href="/safari-pinned-tab.svg" color="#5bbad5">
<meta name="msapplication-TileColor" content="#da532c">
<meta name="theme-color" content="#3079F1">
<link rel="manifest" href="/pwa/manifestv1.json">
<meta itemprop="inLanguage" content="fa-IR">

 <meta content="قیمت دلار,قیمت نقره,قیمت مس,قیمت بنزین,قیمت نفت,قیمت گاز,قیمت ارز,قیمت طلا,قیمت سکه,قیمت آهن,قیمت آهن آلات,نرخ ارز,ارز,طلا,قیمت خودرو,قیمت ماشین,قیمت موبایل,قیمت لپ تاپ,اخبار اقتصادی,سایت اقتصادی,قیمت یورو,قیمت میلگرد,قیمت تیر آهن,قیمت نبشی,قیمت نیم سکه,قیمت ربع سکه,قیمت" name="keywords">  <meta content="قیمت دلار,نرخ ارز,قیمت دلار,قیمت طلا,قیمت سکه,قیمت آه

In [15]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, 'html.parser')


In [ ]:
links_soup_page = soup.find_all('a')
url = set()

for link in links_soup:
     url.add(link.get('href'))

url = list(url)
gold_link = [link for link in url if "طلا" and "سکه" in link]

links_soup


[<a class="showmobile" href="tg://join?invite=ArEAsDwoq3IVfJTHlhWa4g" rel="nofollow noreferrer" target="_blank"><img alt="کانال تلگرام ایران جیب" height="60" loading="lazy" src="https://www.iranjib.ir/images/channel360.png" style="border:none" width="360"/></a>,
 <a href="https://www.iranjib.ir/">صفحه اصلی</a>,
 <a href="https://www.iranjib.ir/showpage/15/تبلیغات/">تبلیغات</a>,
 <a href="https://www.iranjib.ir/contactus/تماس-با-ما/">تماس با ما</a>,
 <a href="https://www.iranjib.ir/?"><img alt="ایران جیب" height="104" loading="lazy" src="https://www.iranjib.ir/images/iranjib-logo.webp" style="border:none" title="ایران جیب" width="254"/></a>,
 <a href="#">اخبار<span class="anchor_down">⌄</span></a>,
 <a href="https://www.iranjib.ir/default/1/اقتصاد/">اقتصاد</a>,
 <a href="https://www.iranjib.ir/default/2/صنعت-و-معدن/">صنعت و معدن</a>,
 <a href="https://www.iranjib.ir/default/3/خودرو/">خودرو</a>,
 <a href="https://www.iranjib.ir/default/4/بورس/">بورس</a>,
 <a href="https://www.iranjib.ir/

In [7]:
gold_link = [link for link in url if "طلا" and "سکه" in link]
gold = gold_link[0]
gold

'https://www.iranjib.ir/shownews/145501/قیمت-جدید-سکه-و-طلا-امروز-سه-شنبه-1405-04-09/'

In [22]:
import jdatetime
import re
from bs4 import BeautifulSoup
import requests
url = "https://www.iranjib.ir/showarchive/0/0"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
pagination = soup.find_all('table',attrs={'class':'pagination'})
page_links = []
with open('gold.txt','w',encoding='utf-8') as f:
    f.write('date,year,month,day,coin,gold\n')

for page in pagination:
    for link in page.find_all("a"):
        page_links.append(link["href"])
for page in page_links:
    response = requests.get(page)
    soup = BeautifulSoup(response.text, 'html.parser')
    links_soup_page = soup.find_all('a')
    url = set()
    for link in links_soup_page:
        url.add(link.get('href'))
    url = list(url)
    gold_link = [link for link in url if "طلا" and "سکه" in link]

    with open('gold.txt','a',encoding='utf-8') as f:
         for link in gold_link:
            try:
                page = requests.get(link)
                page_soup = BeautifulSoup(page.text , "html.parser")
                date_soup = page_soup.find_all('td',attrs={"style":"width:60%"})[0].get_text()
                day, month_name, year = re.search(r'(\d+)\s+(\S+)\s+(\d+)', date_soup).groups()
                months = {
                    "فروردین": 1, "اردیبهشت": 2, "خرداد": 3, "تیر": 4,
                    "مرداد": 5, "شهریور": 6, "مهر": 7, "آبان": 8,
                    "آذر": 9, "دی": 10, "بهمن": 11, "اسفند": 12
                }
                date_Shamsi = jdatetime.date(int(year),int(months[month_name]),int(day)).strftime("%Y/%m/%d")


                table = page_soup.find(attrs={"class":"nij"})
                for row in table.find_all('tr') :
                    if "سکه گرمی" in row.text:
                        price_coin =int( row.find_all('td')[1].get_text(strip=True).replace(',',''))
                    if "طلای 18 عیار" in row.text:
                        price_gold = int( row.find_all('td')[1].get_text(strip=True).replace(',',''))

                f.write(f'{date_Shamsi} , {year} , {month_name} , {day} , {price_coin} , {price_gold}\n') 
         
            except Exception as e:
               print( f'error {link} message {e}')

error https://www.iranjib.ir/shownews/145153/ریزش-سریالی-قیمت-سکه-و-دلار/ message 'NoneType' object has no attribute 'find_all'


In [25]:
import sqlite3
import jdatetime
import re
from bs4 import BeautifulSoup
import requests
conn = sqlite3.Connection('gold_coin.db')
cursor = conn.cursor()
state_Create_table ='''
                              create table if not exists gold 
                                    (id, Date_Shamsi , year , MonthName , Days ,Price_coin , Price_Gold)

'''
cursor.execute(state_Create_table)
state_Insert_table = '''
                                   insert into gold  values (?,?,?,?,?,?,?)

                                   '''
url = "https://www.iranjib.ir/showarchive/0/0"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
pagination = soup.find_all('table',attrs={'class':'pagination'})
page_links = []


for page in pagination:
    for link in page.find_all("a"):
        page_links.append(link["href"])
for page in page_links:
    response = requests.get(page)
    soup = BeautifulSoup(response.text, 'html.parser')
    links_soup_page = soup.find_all('a')
    url = set()
    for link in links_soup_page:
        url.add(link.get('href'))
    url = list(url)
    gold_link = [link for link in url if "طلا" and "سکه" in link]

  
    for i, link in enumerate(gold_link):
           try:
                page = requests.get(link)
                page_soup = BeautifulSoup(page.text , "html.parser")
                date_soup = page_soup.find_all('td',attrs={"style":"width:60%"})[0].get_text()
                day, month_name, year = re.search(r'(\d+)\s+(\S+)\s+(\d+)', date_soup).groups()
                months = {
                    "فروردین": 1, "اردیبهشت": 2, "خرداد": 3, "تیر": 4,
                    "مرداد": 5, "شهریور": 6, "مهر": 7, "آبان": 8,
                    "آذر": 9, "دی": 10, "بهمن": 11, "اسفند": 12
                }
                date_Shamsi = jdatetime.date(int(year),int(months[month_name]),int(day)).strftime("%Y/%m/%d")


                table = page_soup.find(attrs={"class":"nij"})
                for row in table.find_all('tr') :
                    if "سکه گرمی" in row.text:
                        price_coin =int( row.find_all('td')[1].get_text(strip=True).replace(',',''))
                    if "طلای 18 عیار" in row.text:
                        price_gold = int( row.find_all('td')[1].get_text(strip=True).replace(',',''))
                  
                cursor.execute(state_Insert_table,(i,
                                                 date_Shamsi,
                                                 year,
                                                  month_name,
                                                  day,
                                                  price_coin,
                                                  price_gold
                                                           
                                                           
                                                )
                                                           )
                conn.commit
           except Exception as e:
                  print( f'error: {link} message: {e}')


error: https://www.iranjib.ir/shownews/145536/قیمت-جدید-سکه-و-طلا-امروز-چهارشنبه-1405-04-10/ message: database is locked
error: https://www.iranjib.ir/shownews/145501/قیمت-جدید-سکه-و-طلا-امروز-سه-شنبه-1405-04-09/ message: database is locked
error: https://www.iranjib.ir/shownews/145460/قیمت-جدید-سکه-و-طلا-امروز-دوشنبه-1405-04-08/ message: database is locked
error: https://www.iranjib.ir/shownews/145420/قیمت-جدید-سکه-و-طلا-امروز-یکشنبه-1405-04-07/ message: database is locked
error: https://www.iranjib.ir/shownews/145384/قیمت-جدید-سکه-و-طلا-امروز-شنبه-1405-04-06/ message: database is locked
error: https://www.iranjib.ir/shownews/145320/قیمت-جدید-سکه-و-طلا-امروز-سه-شنبه-1405-04-02/ message: database is locked
error: https://www.iranjib.ir/shownews/145255/قیمت-جدید-سکه-و-طلا-امروز-یکشنبه-1405-03-31/ message: database is locked
error: https://www.iranjib.ir/shownews/145294/قیمت-جدید-سکه-و-طلا-امروز-دوشنبه-1405-04-01/ message: database is locked
error: https://www.iranjib.ir/shownews/145224/

KeyboardInterrupt: 

In [9]:
import jdatetime
import re
from bs4 import BeautifulSoup
import requests
url = "https://www.iranjib.ir/showarchive/0/0"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
pagination = soup.find_all('table',attrs={'class':'pagination'})
page_links = []
for page in pagination:
     for link in page.find_all('a'):
          page_links.append(link.get('href'))


In [13]:
link


<a class="jaxlink" href="https://www.iranjib.ir/jax/showarchive.php?p=1&amp;_id=12" wrapper="showarchivetarget"><button class="paginbutton">12</button></a>